In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import os
import scipy.io
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

In [ ]:


class MultiModalTeaDataset(Dataset):
    def __init__(self, image_root, nir_mat_path, transform=None, train=True):
        self.image_root = image_root
        self.nir_mat_path = nir_mat_path
        self.transform = transform
        self.train = train

        self.classes = sorted(os.listdir(image_root))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        self.image_list = self._load_image_paths()

        nir_mat = scipy.io.loadmat(nir_mat_path)
        self.nir_data = nir_mat['nir_data'] if 'nir_data' in nir_mat else list(nir_mat.values())[-1]  # [120, 波段数]

        assert len(self.image_list) == self.nir_data.shape[0], \
            f"图像数量 {len(self.image_list)} 与 NIR 数据 {self.nir_data.shape[0]} 不匹配"

    def _load_image_paths(self):
        img_list = []
        for cls in self.classes:
            cls_path = os.path.join(self.image_root, cls)
            for img_name in sorted(os.listdir(cls_path)):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(cls_path, img_name)
                    img_list.append((img_path, self.class_to_idx[cls]))
        return img_list

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_path, label = self.image_list[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        nir = self.nir_data[idx]  # shape [波段数]
        nir = nir.astype(np.float32)

        return image, nir, label


In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        # 使用ResNet50-MoCo预训练权重
        self.encoder = torch.hub.load('facebookresearch/moco:v1', 'resnet50', pretrained=True)
        self.encoder.fc = nn.Identity()  # 移除分类头，输出[batch, 2048]

    def forward(self, x):
        return self.encoder(x)

class NIR_Encoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten())

    def forward(self, x):
        return self.conv_blocks(x.unsqueeze(1))

class ContrastiveFusion(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.img_proj = nn.Linear(768, dim)
        self.nir_proj = nn.Linear(128, dim)

    def forward(self, img_feat, nir_feat):
        img_proj = self.img_proj(img_feat)
        nir_proj = self.nir_proj(nir_feat)
        return F.normalize(img_proj + nir_proj, dim=-1)

class CrossModalAttention(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.query = nn.Linear(128, embed_dim)
        self.key_value = nn.Linear(768, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=4, batch_first=True)

    def forward(self, nir_feat, img_feat):
        q = self.query(nir_feat).unsqueeze(1)
        kv = self.key_value(img_feat).unsqueeze(1)
        out, _ = self.attn(q, kv, kv)
        return out.squeeze(1)

class MultiModalClassifier(nn.Module):
    def __init__(self, num_classes, fusion='contrastive'):
        super().__init__()
        self.img_encoder = ImageEncoder()
        self.nir_encoder = NIR_Encoder()
        self.fusion_type = fusion
        self.projection_dim = 256

        if fusion == 'contrastive':
            self.fusion = ContrastiveFusion(dim=self.projection_dim)
        elif fusion == 'attention':
            self.fusion = CrossModalAttention(embed_dim=self.projection_dim)

        self.classifier = nn.Sequential(
            nn.Linear(self.projection_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes))

    def forward(self, img, nir):
        img_feat = self.img_encoder(img)
        nir_feat = self.nir_encoder(nir)
        fused_feat = self.fusion(img_feat, nir_feat)
        return self.classifier(fused_feat), fused_feat

# ------------------------------ 辅助函数 ------------------------------

def contrastive_loss(features, temperature=0.1):
    batch_size = features.size(0)
    sim_matrix = F.cosine_similarity(features.unsqueeze(1), features.unsqueeze(0), dim=2)
    mask = torch.eye(batch_size, dtype=torch.bool).to(features.device)
    pos = sim_matrix[mask].view(batch_size, 1)
    neg = sim_matrix[~mask].view(batch_size, -1)
    logits = torch.cat([pos, neg], dim=1)
    labels = torch.zeros(batch_size, dtype=torch.long).to(features.device)
    return F.cross_entropy(logits / temperature, labels)

def progressive_unfreeze(model, stage):
    if stage == 1:
        for param in model.img_encoder.parameters():
            param.requires_grad = False
    elif stage == 2:
        for name, param in model.img_encoder.model.named_parameters():
            if 'blocks.10' in name or 'blocks.11' in name:
                param.requires_grad = True
            else:
                param.requires_grad = False
    else:
        for param in model.img_encoder.parameters():
            param.requires_grad = True

# ------------------------------ 数据增强 ------------------------------

def image_augment():
    return T.Compose([
        T.RandAugment(),
        T.ToTensor()
    ])

def augment_nir(nir_tensor):
    if torch.rand(1) < 0.5:
        nir_tensor += torch.randn_like(nir_tensor) * 0.01
    if torch.rand(1) < 0.5:
        mask_idx = torch.randint(0, nir_tensor.shape[1], (5,))
        nir_tensor[:, mask_idx] = 0
    return nir_tensor

# ------------------------------ 蒸馏 & 量化 ------------------------------

def distill_loss(student_out, teacher_out, T=2):
    return F.kl_div(
        F.log_softmax(student_out / T, dim=1),
        F.softmax(teacher_out / T, dim=1),
        reduction='batchmean') * T * T

def quantize_model(model):
    return torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

# ------------------------------ 可解释性分析 ------------------------------

def plot_attention_weights(attn_map, title='Attention Map'):
    plt.figure(figsize=(6, 5))
    sns.heatmap(attn_map.cpu().detach().numpy(), cmap='viridis')
    plt.title(title)
    plt.show()

def shap_analysis(model, data, background_data):
    explainer = shap.DeepExplainer(model, background_data)
    shap_values = explainer.shap_values(data)
    shap.summary_plot(shap_values, data.cpu().numpy())

# ------------------------------ 评估与混淆矩阵 ------------------------------

def evaluate(model, dataloader, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in dataloader:
            img, nir, label = batch['image'].to(device), batch['nir'].to(device), batch['label'].to(device)
            output, _ = model(img, nir)
            preds.append(torch.argmax(output, dim=1).cpu())
            trues.append(label.cpu())
    preds = torch.cat(preds)
    trues = torch.cat(trues)
    cm = confusion_matrix(trues, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')
    plt.show()
